# PCL Detection — Tasks 3 & 4

**Task 3**: Binary classification — does a paragraph contain Patronizing/Condescending Language (PCL)?  
**Task 4**: Multi-label classification — which of the 7 PCL sub-categories are present?

**Model**: `microsoft/deberta-v3-base` (outperforms RoBERTa-base baseline, F1 = 0.48)  
**Key improvements**: class-weighted loss · threshold tuning · mixed-precision training

**PCL categories (Task 4):**
0. Unbalanced power relations  
1. Shallow solution  
2. Presupposition  
3. Authority voice  
4. Metaphor  
5. Compassion  
6. The poorer the merrier

In [ ]:
# Install required packages — run this cell once, then restart the kernel
# !pip install transformers>=4.35.0 sentencepiece protobuf torch scikit-learn pandas numpy tqdm

# Packages needed and why:
#   transformers  — AutoTokenizer, AutoModelForSequenceClassification
#   sentencepiece — DeBERTa-v3 uses a SentencePiece tokenizer (.spm.model)
#   protobuf      — required by sentencepiece to parse the binary .spm.model file
#
# NOTE: tiktoken is NOT needed — it is for OpenAI/GPT models only

In [ ]:
import os, ast, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, classification_report
)
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"BF16   : {'supported' if bf16_ok else 'NOT supported — will use FP32'}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
CFG = {
    # Model
    'model_name':    'microsoft/deberta-v3-base',
    # Data files
    'main_data':     'dontpatronizeme_pcl.tsv',
    'train_ids':     'train_semeval_parids-labels.csv',
    'dev_ids':       'dev_semeval_parids-labels.csv',
    'test_data':     'task4_test.tsv',
    # Tokenizer
    'max_length':    128,
    # Training
    'batch_size':    16,
    'grad_accum':    2,        # effective batch = 32
    'lr':            2e-5,
    'weight_decay':  0.01,
    'warmup_ratio':  0.10,
    'num_epochs_t3': 5,
    'num_epochs_t4': 5,
    'seed':          42,
    # Outputs
    'task3_model':   'task3_best.pt',
    'task4_model':   'task4_best.pt',
    'task3_preds':   'task3_test_preds.csv',
    'task4_preds':   'task4_test_preds.csv',
}

PCL_CATEGORIES = [
    'Unbalanced power relations',
    'Shallow solution',
    'Presupposition',
    'Authority voice',
    'Metaphor',
    'Compassion',
    'The poorer the merrier',
]

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(CFG['seed'])
print("Config ready.")

## 1. Data Loading

In [ ]:
# ── Load main dataset (dontpatronizeme_pcl.tsv) ──────────────────────────────
# Uses a line-by-line parser that safely skips the 4-row disclaimer header
# and any malformed rows.

def load_main_data(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t', 5)
            if len(parts) == 6 and parts[0].strip().isdigit():
                rows.append({
                    'par_id':    int(parts[0]),
                    'keyword':   parts[2],
                    'country':   parts[3],
                    'text':      parts[4].strip(),
                    'label_raw': int(parts[5]),
                })
    df = pd.DataFrame(rows)
    # Task 3 binary label: PCL if >= 2 annotators flagged it
    df['label_binary'] = (df['label_raw'] >= 2).astype(int)
    return df


def load_split_ids(path):
    """Load par_ids and their 7-element Task-4 multi-label vectors."""
    df = pd.read_csv(path)
    df['par_id'] = df['par_id'].astype(int)
    df['label_list'] = df['label'].apply(ast.literal_eval)
    return df[['par_id', 'label_list']]


def load_test_data(path):
    """Load test set — par_ids start with 't_', no label column, no disclaimer."""
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) >= 5 and parts[0].strip().startswith('t_'):
                rows.append({
                    'par_id':  parts[0].strip(),
                    'keyword': parts[2] if len(parts) > 2 else '',
                    'country': parts[3] if len(parts) > 3 else '',
                    'text':    parts[4].strip() if len(parts) > 4 else '',
                })
    df = pd.DataFrame(rows)
    df = df[df['text'].str.len() > 0].reset_index(drop=True)
    return df


# Load everything
df_main      = load_main_data(CFG['main_data'])
df_train_ids = load_split_ids(CFG['train_ids'])
df_dev_ids   = load_split_ids(CFG['dev_ids'])
df_test      = load_test_data(CFG['test_data'])

print(f"Main dataset : {len(df_main):>6,} rows")
print(f"Train IDs    : {len(df_train_ids):>6,} rows")
print(f"Dev IDs      : {len(df_dev_ids):>6,} rows")
print(f"Test set     : {len(df_test):>6,} rows")

In [ ]:
# ── Merge labels into train / dev splits ─────────────────────────────────────

df_train = df_main[df_main['par_id'].isin(df_train_ids['par_id'])].copy()
df_dev   = df_main[df_main['par_id'].isin(df_dev_ids['par_id'])].copy()

df_train = df_train.merge(df_train_ids, on='par_id', how='left').reset_index(drop=True)
df_dev   = df_dev.merge(df_dev_ids,   on='par_id', how='left').reset_index(drop=True)

# Fill Task-4 labels with all-zeros for samples not in the label CSV
_zero7 = lambda x: x if isinstance(x, list) else [0] * 7
df_train['label_list'] = df_train['label_list'].apply(_zero7)
df_dev['label_list']   = df_dev['label_list'].apply(_zero7)

print(f"Train : {len(df_train):,} samples")
print(f"Dev   : {len(df_dev):,} samples")

# Task 3 class breakdown
n_neg_tr = (df_train['label_binary'] == 0).sum()
n_pos_tr = (df_train['label_binary'] == 1).sum()
n_neg_dv = (df_dev['label_binary'] == 0).sum()
n_pos_dv = (df_dev['label_binary'] == 1).sum()
print(f"\nTask 3 — Train: No-PCL={n_neg_tr:,}  PCL={n_pos_tr:,}  ratio={n_neg_tr/n_pos_tr:.1f}:1")
print(f"Task 3 — Dev  : No-PCL={n_neg_dv:,}  PCL={n_pos_dv:,}")

In [ ]:
# ── Data verification ─────────────────────────────────────────────────────────

# 1. No overlap between train and dev
overlap = set(df_train['par_id']) & set(df_dev['par_id'])
print(f"Train/dev par_id overlap (should be 0): {len(overlap)}")

# 2. Task-4 label array stats
train_arr = np.array(df_train['label_list'].tolist(), dtype=np.float32)
dev_arr   = np.array(df_dev['label_list'].tolist(),   dtype=np.float32)

print(f"\nTask 4 per-category positive counts (train) — {len(train_arr):,} samples total:")
for i, cat in enumerate(PCL_CATEGORIES):
    cnt = int(train_arr[:, i].sum())
    pct = train_arr[:, i].mean() * 100
    print(f"  [{i}] {cat:<35s} : {cnt:>4} ({pct:.1f}%)")

# 3. Quick text sample
print("\nSample row:")
print(df_train[df_train['label_binary'] == 1].iloc[0][['text', 'label_binary', 'label_list']].to_string())

## 2. Dataset Class & Training Utilities

In [ ]:
# ── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
print(f"Tokenizer loaded: {CFG['model_name']}")


# ── Dataset class (shared for binary and multi-label) ───────────────────────
class PCLDataset(Dataset):
    """
    Works for both Task 3 (labels = list of int scalars)
    and Task 4 (labels = list of 7-element lists).
    """
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts      = [str(t) for t in texts]
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        lbl = self.labels[idx]
        if isinstance(lbl, (list, np.ndarray)):
            item['labels'] = torch.tensor(lbl, dtype=torch.float)
        else:
            item['labels'] = torch.tensor(float(lbl), dtype=torch.float)
        return item

In [ ]:
# ── Training / evaluation / prediction helpers ───────────────────────────────
# Pure FP32: DeBERTa-v3's disentangled attention + StableDropout
# are numerically unstable in BF16 on Blackwell (RTX 50-series).
# FP32 is still fast enough on RTX 5070 for this dataset size.

def run_epoch(model, loader, optimizer, scheduler, criterion,
              grad_accum, is_multilabel=False):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader), total=len(loader), desc='Train', leave=True)
    for step, batch in pbar:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        extra = {}
        if 'token_type_ids' in batch:
            extra['token_type_ids'] = batch['token_type_ids'].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **extra
        ).logits

        if not is_multilabel:
            logits = logits.squeeze(-1)

        loss = criterion(logits, labels) / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        pbar.set_postfix({'loss': f'{total_loss / (step + 1):.4f}'})

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, threshold=0.5, is_multilabel=False):
    model.eval()
    all_logits, all_labels = [], []

    for batch in tqdm(loader, desc='Eval', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        extra = {}
        if 'token_type_ids' in batch:
            extra['token_type_ids'] = batch['token_type_ids'].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **extra
        ).logits

        if not is_multilabel:
            logits = logits.squeeze(-1)

        all_logits.append(logits.cpu().float().numpy())
        all_labels.append(batch['labels'].numpy())

    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    probs = 1.0 / (1.0 + np.exp(-all_logits))
    preds = (probs >= threshold).astype(int)

    if is_multilabel:
        f1 = f1_score(all_labels.astype(int), preds, average='macro', zero_division=0)
    else:
        f1 = f1_score(all_labels.astype(int), preds, zero_division=0)

    return f1, probs, all_labels


def tune_threshold(probs, labels, is_multilabel=False):
    grid = np.linspace(0.05, 0.95, 91)

    if is_multilabel:
        best_thresholds = []
        for i in range(labels.shape[1]):
            best_t, best_f = 0.5, 0.0
            for t in grid:
                p = (probs[:, i] >= t).astype(int)
                f = f1_score(labels[:, i].astype(int), p, zero_division=0)
                if f > best_f:
                    best_f, best_t = f, t
            best_thresholds.append(round(float(best_t), 3))
        return best_thresholds
    else:
        best_t, best_f = 0.5, 0.0
        for t in grid:
            p = (probs >= t).astype(int)
            f = f1_score(labels.astype(int), p, zero_division=0)
            if f > best_f:
                best_f, best_t = f, t
        print(f"Optimal threshold: {best_t:.3f}  →  F1 = {best_f:.4f}")
        return float(best_t)


@torch.no_grad()
def predict(model, loader, threshold, is_multilabel=False):
    model.eval()
    all_probs = []

    for batch in tqdm(loader, desc='Predict', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        extra = {}
        if 'token_type_ids' in batch:
            extra['token_type_ids'] = batch['token_type_ids'].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **extra
        ).logits

        if not is_multilabel:
            logits = logits.squeeze(-1)

        all_probs.append(torch.sigmoid(logits).cpu().float().numpy())

    all_probs = np.concatenate(all_probs, axis=0)

    if is_multilabel:
        preds = np.zeros_like(all_probs, dtype=int)
        for i, t in enumerate(threshold):
            preds[:, i] = (all_probs[:, i] >= t).astype(int)
    else:
        preds = (all_probs >= threshold).astype(int)

    return preds, all_probs


print("Utilities defined (pure FP32).")

---
## 3. Task 3 — Binary PCL Detection

**Baseline**: RoBERTa-base, standard loss, threshold=0.5 → **F1 = 0.48**

**Our approach**:
- DeBERTa-v3-base (significantly stronger encoder)
- `BCEWithLogitsLoss` with `pos_weight ≈ 9.55` (counteracts 1:9.5 class imbalance)
- Threshold search on dev set to maximise F1
- Mixed-precision (FP16) training with gradient accumulation

In [ ]:
# ── Task 3: data loaders ─────────────────────────────────────────────────────
train_ds_t3 = PCLDataset(df_train['text'].tolist(),
                          df_train['label_binary'].tolist(),
                          tokenizer, CFG['max_length'])
dev_ds_t3   = PCLDataset(df_dev['text'].tolist(),
                          df_dev['label_binary'].tolist(),
                          tokenizer, CFG['max_length'])
test_ds_t3  = PCLDataset(df_test['text'].tolist(),
                          [0] * len(df_test),
                          tokenizer, CFG['max_length'])

train_dl_t3 = DataLoader(train_ds_t3, batch_size=CFG['batch_size'],
                          shuffle=True,  num_workers=0, pin_memory=True)
dev_dl_t3   = DataLoader(dev_ds_t3,   batch_size=CFG['batch_size'] * 2,
                          shuffle=False, num_workers=0, pin_memory=True)
test_dl_t3  = DataLoader(test_ds_t3,  batch_size=CFG['batch_size'] * 2,
                          shuffle=False, num_workers=0)

# Class-weighted loss
n_neg_t3 = int((df_train['label_binary'] == 0).sum())
n_pos_t3 = int((df_train['label_binary'] == 1).sum())
pos_weight_t3 = torch.tensor([n_neg_t3 / n_pos_t3], dtype=torch.float).to(device)
criterion_t3  = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t3)
print(f"pos_weight_t3 = {pos_weight_t3.item():.2f}  (No-PCL={n_neg_t3}, PCL={n_pos_t3})")

# Model — num_labels=1 → single logit, BCEWithLogitsLoss
set_seed(CFG['seed'])
model_t3 = AutoModelForSequenceClassification.from_pretrained(
    CFG['model_name'], num_labels=1
).to(device)

# Optimiser and scheduler
total_steps_t3  = (len(train_dl_t3) // CFG['grad_accum']) * CFG['num_epochs_t3']
warmup_steps_t3 = int(total_steps_t3 * CFG['warmup_ratio'])

optimizer_t3 = AdamW(model_t3.parameters(),
                      lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler_t3 = get_linear_schedule_with_warmup(
    optimizer_t3,
    num_warmup_steps=warmup_steps_t3,
    num_training_steps=total_steps_t3
)

print(f"Total steps: {total_steps_t3} | Warmup: {warmup_steps_t3}")

In [ ]:
# ── NaN Diagnostic — run this BEFORE cell-t3-train ───────────────────────────
# Tells us exactly which step introduces NaN

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"Param dtype: {next(model_t3.parameters()).dtype}")

model_t3.train()
batch = next(iter(train_dl_t3))
inp = batch['input_ids'].to(device)
msk = batch['attention_mask'].to(device)
lbl = batch['labels'].to(device)

print(f"\nLabels  — NaN? {torch.isnan(lbl).any()}  range: [{lbl.min():.0f}, {lbl.max():.0f}]")
print(f"Input IDs — shape: {inp.shape}")

# 1) eval mode, no grad
model_t3.eval()
with torch.no_grad():
    out_eval = model_t3(input_ids=inp, attention_mask=msk).logits.squeeze(-1)
print(f"\n[1] eval+no_grad  → NaN? {torch.isnan(out_eval).any()}  | values: {out_eval[:4].tolist()}")

# 2) train mode, no grad
model_t3.train()
with torch.no_grad():
    out_train_ng = model_t3(input_ids=inp, attention_mask=msk).logits.squeeze(-1)
print(f"[2] train+no_grad → NaN? {torch.isnan(out_train_ng).any()}  | values: {out_train_ng[:4].tolist()}")

# 3) train mode, with grad
out_train_g = model_t3(input_ids=inp, attention_mask=msk).logits.squeeze(-1)
print(f"[3] train+grad    → NaN? {torch.isnan(out_train_g).any()}  | values: {out_train_g[:4].tolist()}")

# 4) loss
loss_dbg = criterion_t3(out_train_g, lbl)
print(f"[4] loss          → {loss_dbg.item():.6f}")

# 5) backward + grad check
loss_dbg.backward()
nan_grads = [(n, p) for n, p in model_t3.named_parameters()
             if p.grad is not None and torch.isnan(p.grad).any()]
print(f"[5] NaN gradients after backward: {len(nan_grads)}")
if nan_grads:
    print(f"    First offender: {nan_grads[0][0]}")
model_t3.zero_grad()

print("\n=== Diagnostic done — paste output above ===")

In [ ]:
# ── Task 3: training loop ────────────────────────────────────────────────────
best_f1_t3     = 0.0
best_probs_t3  = None
best_labels_t3 = None

for epoch in range(CFG['num_epochs_t3']):
    print(f"\n── Epoch {epoch + 1} / {CFG['num_epochs_t3']} ──")
    train_loss = run_epoch(
        model_t3, train_dl_t3, optimizer_t3, scheduler_t3,
        criterion_t3, CFG['grad_accum'], is_multilabel=False
    )
    f1, probs, labels = evaluate(model_t3, dev_dl_t3, threshold=0.5, is_multilabel=False)
    print(f"   Train loss: {train_loss:.4f}  |  Dev F1 (t=0.5): {f1:.4f}")

    if f1 > best_f1_t3:
        best_f1_t3     = f1
        best_probs_t3  = probs.copy()
        best_labels_t3 = labels.copy()
        torch.save(model_t3.state_dict(), CFG['task3_model'])
        print(f"   ✓ New best  (F1={best_f1_t3:.4f}) — saved to {CFG['task3_model']}")

print(f"\nBest dev F1 before threshold tuning: {best_f1_t3:.4f}")

In [ ]:
# ── Task 3: threshold tuning + final evaluation ──────────────────────────────

# Load best checkpoint and get fresh probs
model_t3.load_state_dict(torch.load(CFG['task3_model'], map_location=device))
_, best_probs_t3, best_labels_t3 = evaluate(model_t3, dev_dl_t3, threshold=0.5)

print("Searching optimal threshold on dev set ...")
best_thresh_t3 = tune_threshold(best_probs_t3, best_labels_t3, is_multilabel=False)

preds_dev_t3 = (best_probs_t3 >= best_thresh_t3).astype(int)

print(f"\n{'=' * 55}")
print(f"  TASK 3 FINAL RESULTS  (threshold = {best_thresh_t3:.3f})")
print(f"{'=' * 55}")
print(f"  Baseline (RoBERTa-base, t=0.5)   F1 = 0.4800")
t3_f1 = f1_score(best_labels_t3.astype(int), preds_dev_t3)
print(f"  Our model (DeBERTa-v3-base)       F1 = {t3_f1:.4f}")
print(f"  Improvement: +{t3_f1 - 0.48:.4f}")
print()
print(classification_report(
    best_labels_t3.astype(int), preds_dev_t3,
    target_names=['No-PCL', 'PCL'], digits=4
))

In [ ]:
# ── Task 3: test set predictions ─────────────────────────────────────────────
preds_test_t3, probs_test_t3 = predict(
    model_t3, test_dl_t3, threshold=best_thresh_t3, is_multilabel=False
)

out_t3 = pd.DataFrame({
    'par_id':     df_test['par_id'].values,
    'label':      preds_test_t3.tolist(),
    'prob_pcl':   probs_test_t3.round(4).tolist(),
})
out_t3.to_csv(CFG['task3_preds'], index=False)

print(f"Saved → {CFG['task3_preds']}")
print(f"Predicted PCL: {preds_test_t3.sum()} / {len(preds_test_t3)}  ({preds_test_t3.mean()*100:.1f}%)")
print(out_t3.head(10).to_string(index=False))

---
## 4. Task 4 — Multi-label PCL Sub-category Classification

Predict which of the 7 PCL sub-categories are present in each paragraph.

**Approach**:
- Same DeBERTa-v3-base encoder, 7 output logits
- `BCEWithLogitsLoss` with **per-label** `pos_weight` (each category has a different imbalance ratio)
- Per-label threshold search on dev set

In [ ]:
# ── Task 4: data loaders ─────────────────────────────────────────────────────
train_ds_t4 = PCLDataset(df_train['text'].tolist(),
                          df_train['label_list'].tolist(),
                          tokenizer, CFG['max_length'])
dev_ds_t4   = PCLDataset(df_dev['text'].tolist(),
                          df_dev['label_list'].tolist(),
                          tokenizer, CFG['max_length'])
test_ds_t4  = PCLDataset(df_test['text'].tolist(),
                          [[0] * 7] * len(df_test),
                          tokenizer, CFG['max_length'])

train_dl_t4 = DataLoader(train_ds_t4, batch_size=CFG['batch_size'],
                          shuffle=True,  num_workers=0, pin_memory=True)
dev_dl_t4   = DataLoader(dev_ds_t4,   batch_size=CFG['batch_size'] * 2,
                          shuffle=False, num_workers=0, pin_memory=True)
test_dl_t4  = DataLoader(test_ds_t4,  batch_size=CFG['batch_size'] * 2,
                          shuffle=False, num_workers=0)

# Per-label class weights
pos_counts   = train_arr.sum(axis=0).clip(min=1)
neg_counts   = len(train_arr) - pos_counts
pw_t4        = torch.tensor(neg_counts / pos_counts, dtype=torch.float).to(device)
criterion_t4 = nn.BCEWithLogitsLoss(pos_weight=pw_t4)

print("Per-label pos_weights:")
for cat, w in zip(PCL_CATEGORIES, pw_t4.cpu().numpy()):
    print(f"  {cat:<35s}: {w:.1f}")

# Model — num_labels=7 for multi-label BCEWithLogitsLoss
set_seed(CFG['seed'])
model_t4 = AutoModelForSequenceClassification.from_pretrained(
    CFG['model_name'], num_labels=7
).to(device)

total_steps_t4  = (len(train_dl_t4) // CFG['grad_accum']) * CFG['num_epochs_t4']
warmup_steps_t4 = int(total_steps_t4 * CFG['warmup_ratio'])

optimizer_t4 = AdamW(model_t4.parameters(),
                      lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler_t4 = get_linear_schedule_with_warmup(
    optimizer_t4,
    num_warmup_steps=warmup_steps_t4,
    num_training_steps=total_steps_t4
)

print(f"\nTotal steps: {total_steps_t4} | Warmup: {warmup_steps_t4}")

In [ ]:
# ── Task 4: training loop ────────────────────────────────────────────────────
best_f1_t4     = 0.0
best_probs_t4  = None
best_labels_t4 = None

for epoch in range(CFG['num_epochs_t4']):
    print(f"\n── Epoch {epoch + 1} / {CFG['num_epochs_t4']} ──")
    train_loss = run_epoch(
        model_t4, train_dl_t4, optimizer_t4, scheduler_t4,
        criterion_t4, CFG['grad_accum'], is_multilabel=True
    )
    f1, probs, labels = evaluate(model_t4, dev_dl_t4, threshold=0.5, is_multilabel=True)
    print(f"   Train loss: {train_loss:.4f}  |  Dev Macro-F1 (t=0.5): {f1:.4f}")

    if f1 > best_f1_t4:
        best_f1_t4     = f1
        best_probs_t4  = probs.copy()
        best_labels_t4 = labels.copy()
        torch.save(model_t4.state_dict(), CFG['task4_model'])
        print(f"   ✓ New best  (Macro-F1={best_f1_t4:.4f}) — saved to {CFG['task4_model']}")

print(f"\nBest dev Macro-F1 before threshold tuning: {best_f1_t4:.4f}")

In [ ]:
# ── Task 4: per-label threshold tuning + final evaluation ────────────────────

model_t4.load_state_dict(torch.load(CFG['task4_model'], map_location=device))
_, best_probs_t4, best_labels_t4 = evaluate(
    model_t4, dev_dl_t4, threshold=0.5, is_multilabel=True
)

print("Per-label threshold search ...")
best_thresholds_t4 = tune_threshold(best_probs_t4, best_labels_t4, is_multilabel=True)

# Apply per-label thresholds
preds_dev_t4 = np.zeros_like(best_probs_t4, dtype=int)
for i, t in enumerate(best_thresholds_t4):
    preds_dev_t4[:, i] = (best_probs_t4[:, i] >= t).astype(int)

print(f"\n{'=' * 55}")
print(f"  TASK 4 FINAL RESULTS")
print(f"{'=' * 55}")
print(f"  {'Category':<35s}  {'t':>5}  {'P':>6}  {'R':>6}  {'F1':>6}")
print(f"  {'-'*35}  {'-'*5}  {'-'*6}  {'-'*6}  {'-'*6}")
f1s = []
for i, cat in enumerate(PCL_CATEGORIES):
    lbl_i = best_labels_t4[:, i].astype(int)
    prd_i = preds_dev_t4[:, i]
    p = precision_score(lbl_i, prd_i, zero_division=0)
    r = recall_score(lbl_i, prd_i, zero_division=0)
    f = f1_score(lbl_i, prd_i, zero_division=0)
    f1s.append(f)
    print(f"  [{i}] {cat:<33s}  {best_thresholds_t4[i]:5.3f}  {p:6.3f}  {r:6.3f}  {f:6.4f}")

macro_f1_t4 = np.mean(f1s)
print(f"\n  Macro-F1: {macro_f1_t4:.4f}")

In [ ]:
# ── Task 4: test set predictions ─────────────────────────────────────────────
preds_test_t4, _ = predict(
    model_t4, test_dl_t4, threshold=best_thresholds_t4, is_multilabel=True
)

out_t4 = pd.DataFrame(preds_test_t4, columns=PCL_CATEGORIES)
out_t4.insert(0, 'par_id', df_test['par_id'].values)
out_t4.to_csv(CFG['task4_preds'], index=False)

print(f"Saved → {CFG['task4_preds']}")
print(f"\nPredicted positives per category:")
for i, cat in enumerate(PCL_CATEGORIES):
    cnt = int(preds_test_t4[:, i].sum())
    pct = preds_test_t4[:, i].mean() * 100
    print(f"  [{i}] {cat:<35s}: {cnt:>4} ({pct:.1f}%)")
print(out_t4.head(8).to_string(index=False))

---
## 5. Summary

| Task | Model | Dev F1 | Notes |
|------|-------|--------|-------|
| Task 3 | RoBERTa-base (baseline) | 0.48 | Standard loss, t=0.5 |
| Task 3 | **DeBERTa-v3-base (ours)** | **see above** | Class-weighted BCELoss + threshold tuning |
| Task 4 | **DeBERTa-v3-base (ours)** | **see above** | Per-label BCELoss + per-label thresholds |

### Why DeBERTa-v3-base outperforms RoBERTa-base
- Uses **disentangled attention** (separate position and content embeddings) — better for syntactic/semantic nuance
- Pre-trained with **ELECTRA-style replaced token detection** on top of MLM — richer representations
- Stronger on GLUE/SuperGLUE benchmarks at the same parameter count (~86M)

### Output files
| File | Contents |
|------|----------|
| `task3_test_preds.csv` | `par_id`, binary `label` (0/1), `prob_pcl` |
| `task4_test_preds.csv` | `par_id`, one column per PCL sub-category (0/1) |
| `task3_best.pt` | Task 3 model weights |
| `task4_best.pt` | Task 4 model weights |